# 06. 지도에서 구조 확인하기

            앞 노트북들은 edge를 선으로 계산했습니다. 마지막으로 저장된 실제 route geometry를 이용해 지도 HTML을 만듭니다.

            이 노트북도 API를 호출하지 않습니다. 이미 저장된 `route_paths.geojson`만 사용합니다.


## 오늘 사용할 말

- graph(그래프): 점과 선으로 이루어진 연결 구조
- node(꼭짓점): 지도 위의 역 후보 지점
- edge(변): 두 지점을 연결하는 하나의 경로
- weight(가중치): 어떤 edge가 좋은지 나쁜지 판단하는 점수
- normalization(정규화): 서로 단위가 다른 값을 비교 가능한 점수로 바꾸는 일
- MST, minimum spanning tree(최소신장수형도): 모든 node를 연결하되 총 비용을 작게 만드는 기본 구조
- shortest path(최단경로): graph 안에서 두 node 사이를 가장 짧게 가는 경로
- stretch(우회율): 선택한 구조에서 얼마나 돌아가는지 나타내는 값
- t-spanner(t-스패너): 너무 많이 돌아가지 않도록 edge를 추가하는 방법


## 1. 준비하기

04번에서 만든 구조별 edge 목록을 읽습니다.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
for path in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (path / "analysis/student_helpers.py").is_file():
        PROJECT_ROOT = path
        break
    if (path / "student_helpers.py").is_file():
        PROJECT_ROOT = path
        break

if (PROJECT_ROOT / "analysis").is_dir():
    sys.path.insert(0, str(PROJECT_ROOT / "analysis"))
else:
    sys.path.insert(0, str(PROJECT_ROOT))

from student_helpers import *

OUT = output_dir(PROJECT_ROOT)
print("작업 폴더:", PROJECT_ROOT)
print("결과 저장 폴더:", OUT)


In [ ]:
stations, _ = load_current_data(PROJECT_ROOT)
structure_edges = read_csv(OUT / "04_structure_edges.csv")
for edge in structure_edges:
    for key in ["pair_order", "from_order", "to_order"]:
        edge[key] = int(edge[key])
    for key in numeric_edge_columns():
        if key in edge:
            edge[key] = float(edge[key])

summary = []
for structure in sorted({edge["structure"] for edge in structure_edges}):
    summary.append({"structure": structure, "edge_count": sum(1 for edge in structure_edges if edge["structure"] == structure)})
write_csv(OUT / "06_algorithm_structure_summary.csv", summary)
print_table(summary, ["structure", "edge_count"], limit=10)


## 2. 실제 도로 형상 가져오기

`route_paths.geojson`에는 네이버 Directions 결과에서 저장해 둔 실제 도로 polyline이 들어 있습니다.

이 파일이 있으면 지도 HTML을 만들고, 학생 배포 ZIP처럼 없으면 이 단계는 건너뜁니다.


In [ ]:
route_geojson_path = PROJECT_ROOT / "data/routes/route_paths.geojson"

if route_geojson_path.is_file():
    route_geojson = json.loads(route_geojson_path.read_text(encoding="utf-8"))
    feature_by_pair = {feature["properties"]["pair_id"]: feature for feature in route_geojson["features"]}
    features = []
    for edge in structure_edges:
        feature = dict(feature_by_pair[edge["pair_id"]])
        props = dict(feature["properties"])
        props.update({"structure": edge["structure"], "scenario_cost": edge["scenario_cost"], "distance_km": edge["distance_km"]})
        feature["properties"] = props
        features.append(feature)
    audit = {"status": "PASS", "route_feature_count": len(route_geojson["features"]), "used_feature_count": len(features), "api_calls": 0, "geometry_source": "data/routes/route_paths.geojson"}
else:
    features = []
    audit = {"status": "SKIPPED", "route_feature_count": 0, "used_feature_count": 0, "api_calls": 0, "geometry_source": "not included in student ZIP"}

write_json(OUT / "06_route_geometry_audit.json", audit)
audit


## 3. 지도 HTML 만들기

지도는 브라우저에서 열 수 있는 HTML 파일입니다.

생각해볼 질문:

- 직선으로 봤을 때와 실제 도로 모양으로 봤을 때 느낌이 다른 edge가 있나요?
- edge 수가 많은 구조가 지도에서는 어떻게 보이나요?


In [ ]:
if features:
    station_payload = json.dumps(stations, ensure_ascii=False)
    route_payload = json.dumps({"type": "FeatureCollection", "features": features}, ensure_ascii=False)
    html_lines = [
        '<!doctype html><html lang="ko"><head><meta charset="utf-8"><title>Siheung route structure map</title>',
        '<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css">',
        '<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>',
        '<style>html,body,#map{height:100%;margin:0}.panel{position:absolute;right:12px;top:12px;z-index:500;background:white;padding:10px;border:1px solid #ddd;font-family:sans-serif;font-size:13px}</style>',
        '</head><body><div id="map"></div><div class="panel"><b>Structure layers</b><div id="layers"></div></div>',
        '<script>',
        'const stations = ' + station_payload + ';',
        'const routes = ' + route_payload + ';',
        "const map = L.map('map').setView([37.38,126.78], 11);",
        "L.tileLayer('https://tile.openstreetmap.org/{z}/{x}/{y}.png', {maxZoom: 19, attribution: '&copy; OpenStreetMap'}).addTo(map);",
        "const colors = {CANDIDATE_GRAPH:'#64748b',CURRENT_03_CANDIDATE:'#2563eb',MST:'#16a34a',T2_SPANNER:'#f97316',DEGREE_LIMIT_3:'#dc2626',DEGREE_LIMIT_4:'#7c3aed'};",
        'const groups = {};',
        "for (const feature of routes.features) { const s = feature.properties.structure; if (!groups[s]) groups[s] = L.layerGroup(); L.geoJSON(feature, {style: {color: colors[s] || '#111827', weight: 4, opacity: 0.72}}).bindTooltip(`${s} ${feature.properties.pair_id}`).addTo(groups[s]); }",
        "for (const s of stations) L.circleMarker([+s.latitude,+s.longitude], {radius:4,color:'#111827',fillColor:'#ef4444',fillOpacity:.9}).bindTooltip(`${s.station_id} ${s.station_name}`).addTo(map);",
        "for (const [name, group] of Object.entries(groups)) { const id = 'layer_' + name; const label = document.createElement('label'); label.innerHTML = `<input type='checkbox' id='${id}'> ${name}`; document.getElementById('layers').appendChild(label); document.getElementById('layers').appendChild(document.createElement('br')); document.getElementById(id).addEventListener('change', e => e.target.checked ? group.addTo(map) : group.removeFrom(map)); }",
        'if (groups.CURRENT_03_CANDIDATE) groups.CURRENT_03_CANDIDATE.addTo(map);',
        '</script></body></html>',
    ]
    html = "\n".join(html_lines)
    (OUT / "06_algorithm_route_maps.html").write_text(html, encoding="utf-8")
    print("지도 HTML 저장:", OUT / "06_algorithm_route_maps.html")
else:
    print("route_paths.geojson이 없어 지도 HTML 생성을 건너뜁니다.")
